In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
import math
import seaborn as sns


try:
    ee.Initialize(project='replicating-paper')
    print("Google Earth Engine Initialized successfully.")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='replicating-paper')

In [ ]:
REGIONS = {
    'sanjay_van': {
        'name': 'Sanjay Van, Delhi',

        'bounds': [77.17, 28.52, 77.18, 28.54],
        'description': 'Urban Dry Deciduous / Scrub'
    },
    'sundarbans': {
        'name': 'Sundarbans Mangroves',
        'bounds': [88.80, 21.80, 89.00, 22.00],
        'description': 'Coastal Mangrove / Tidal'
    },
    'western_ghats': {
        'name': 'Western Ghats (Nilgiris)',
        'bounds': [76.50, 11.30, 76.70, 11.50],
        'description': 'Tropical Evergreen / Montane'
    },
    'himalayas': {
        'name': 'Great Himalayan National Park',
        'bounds': [77.50, 31.70, 77.70, 31.90],
        'description': 'Sub-alpine / Coniferous'
    }
}

CURRENT_REGION_KEY = 'sanjay_van'

# 3-year window to create a robust "Synthetic Year" free of clouds
START_DATE = '2022-01-01'
END_DATE = '2024-12-31'
SCALE = 10

print(f"Target Region: {CURRENT_REGION_KEY}")
print(f"Time Window: {START_DATE} to {END_DATE}")
print(f"Analysis Scale: {SCALE} meters")


roi_info = REGIONS[CURRENT_REGION_KEY]
roi = ee.Geometry.Rectangle(roi_info['bounds'])
print(f"Analysis Focus: {roi_info['name']} ({roi_info['description']})")

In [ ]:
# @title Definitive Data Inventory
data_inventory = [
    {"Dataset": "Sentinel-2 (Harmonized)", "GEE_ID": "COPERNICUS/S2_SR_HARMONIZED", "Resolution": "10m", "Purpose": "Primary Phenology (NDVI, NIRv), Seasonal Cycles", "Temporal Range": "2018 - 2024"},
    {"Dataset": "Dynamic World V1", "GEE_ID": "GOOGLE/DYNAMICWORLD/V1", "Resolution": "10m", "Purpose": "Forest Persistence (Tree cover probability)", "Temporal Range": "2018 - 2024"},
    {"Dataset": "Global Canopy Height (ETH)", "GEE_ID": "users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1", "Resolution": "10m", "Purpose": "Vertical Structure (Tree Height)", "Temporal Range": "2020 (Baseline)"},
    {"Dataset": "Sentinel-1 SAR (C-Band)", "GEE_ID": "COPERNICUS/S1_GRD", "Resolution": "10m", "Purpose": "Canopy Texture & Leaf Volume", "Temporal Range": "2023 - 2024"},
    {"Dataset": "ALOS PALSAR-2 (L-Band)", "GEE_ID": "JAXA/ALOS/PALSAR/YEARLY/SAR", "Resolution": "25m", "Purpose": "Woody Biomass (Trunks/Stems)", "Temporal Range": "2015 - 2024"},
    {"Dataset": "NASADEM (SRTM)", "GEE_ID": "NASA/NASADEM_HGT/001", "Resolution": "30m", "Purpose": "Topography (Elevation, Slope, Aspect)", "Temporal Range": "2000 (Static)"},
    {"Dataset": "WorldClim Bioclim V1", "GEE_ID": "WORLDCLIM/V1/BIO", "Resolution": "1km", "Purpose": "Climate Limits (Temp/Rainfall)", "Temporal Range": "1960-1990"},
    {"Dataset": "CHIRPS Pentad", "GEE_ID": "UCSB-CHG/CHIRPS/PENTAD", "Resolution": "5km", "Purpose": "Water Stress (Drought/Wetness)", "Temporal Range": "2000 - 2024"},
    {"Dataset": "OpenLandMap Soil", "GEE_ID": "OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02", "Resolution": "250m", "Purpose": "Soil Texture/Clay Content", "Temporal Range": "1950 - 2017"},
    {"Dataset": "JRC Global Surface Water", "GEE_ID": "JRC/GSW1_4/GlobalSurfaceWater", "Resolution": "30m", "Purpose": "Hydrology Seasonality", "Temporal Range": "1984 - 2021"},
    {"Dataset": "VIIRS Nighttime Lights", "GEE_ID": "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG", "Resolution": "460m", "Purpose": "Human Pressure (Urbanization)", "Temporal Range": "2014 - 2024"},
    {"Dataset": "Hansen Global Forest Change", "GEE_ID": "UMD/hansen/global_forest_change_2023_v1_11", "Resolution": "30m", "Purpose": "Forest Loss History", "Temporal Range": "2000 - 2023"},
    {"Dataset": "MODIS Burned Area", "GEE_ID": "MODIS/061/MCD64A1", "Resolution": "500m", "Purpose": "Fire History (Years since burn)", "Temporal Range": "2001 - 2023"}
]

df_inventory = pd.DataFrame(data_inventory)

print(f"--- Definitive Data Inventory: {len(df_inventory)} Sources Loaded ---")
display(df_inventory.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    dict(selector='th', props=[('text-align', 'left')])
]))

In [ ]:
# --- UNIFIED MASTER FEATURE EXTRACTION FUNCTION ---
def get_comprehensive_feature_stack(region, start_year, end_year):
    print(f"Building Master Stack for {start_year}-{end_year}...")

    # 1. PHENOLOGY & HARMONICS (Sentinel-2)
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(region) \
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31') \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))

    def add_indices_and_time(img):
        ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
        nirv = ndvi.multiply(img.select('B8').divide(10000)).rename('NIRv')

        # Calculation results in an ee.Number
        time_radians = img.date().difference(ee.Date(f'{start_year}-01-01'), 'year').multiply(2 * math.pi)

        # FIX: Convert ee.Number to ee.Image so we can rename and use as bands
        time_band = ee.Image(time_radians).toFloat()

        return img.addBands([
            ndvi,
            nirv,
            time_band.rename('t'),
            ee.Image.constant(1).rename('constant'),
            time_band.cos().rename('cos'),
            time_band.sin().rename('sin')
        ])

    s2_processed = s2.map(add_indices_and_time)

    def fit_harmonics(collection, dep_var):
        independents = ['constant', 't', 'cos', 'sin']
        trend = collection.select(independents + [dep_var]).reduce(ee.Reducer.linearRegression(4, 1))

        # Extract coefficients and rename them properly
        coeffs = trend.select('coefficients').arrayFlatten([
            [f'{dep_var}_mean', f'{dep_var}_trend', f'{dep_var}_beta_cos', f'{dep_var}_beta_sin'], ['dummy']])

        clean_names = [f'{dep_var}_mean', f'{dep_var}_trend', f'{dep_var}_beta_cos', f'{dep_var}_beta_sin']
        coeffs = coeffs.select([f'{n}_dummy' for n in clean_names], clean_names)

        # Calculate Amplitude and Phase
        amp = coeffs.select(f'{dep_var}_beta_cos').hypot(coeffs.select(f'{dep_var}_beta_sin')).rename(f'{dep_var}_Amp')
        phase = coeffs.select(f'{dep_var}_beta_sin').atan2(coeffs.select(f'{dep_var}_beta_cos')).rename(f'{dep_var}_Phase')
        return coeffs.addBands([amp, phase])

    phenology_stack = fit_harmonics(s2_processed, 'NDVI').addBands(fit_harmonics(s2_processed, 'NIRv'))

    # 2. FOREST STABILITY (Dynamic World)
    dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(region).filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
    stability_stack = dw.select('trees').mean().rename('Tree_Prob_Mean') \
        .addBands(dw.select('trees').reduce(ee.Reducer.stdDev()).rename('Tree_Prob_StdDev')) \
        .addBands(dw.select('label').mode().rename('Dominant_Land_Class'))

    # 3. PHYSICAL STRUCTURE & DISTURBANCE (S1, Canopy, Hansen, MODIS)
    current_year = datetime.datetime.now().year
    ch = ee.Image("users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1").clip(region).select('b1').rename('Canopy_Height').unmask(0)
    ch_texture = ch.reduceNeighborhood(reducer=ee.Reducer.stdDev(), kernel=ee.Kernel.circle(90, 'meters')).rename('Canopy_Texture')

    s1_vh = ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(region).filterDate('2023-01-01', '2024-01-01') \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')).select('VH').mean().rename('S1_VH_Mean')

    palsar_hv = ee.ImageCollection("JAXA/ALOS/PALSAR/YEARLY/SAR").filterBounds(region).first().select('HV').rename('PALSAR_HV')

    # Hansen Loss & MODIS Fire History
    gfc = ee.Image("UMD/hansen/global_forest_change_2023_v1_11").clip(region)
    years_since_loss = ee.Image.constant(current_year).subtract(gfc.select('lossyear').add(2000)) \
        .updateMask(gfc.select('lossyear').gt(0)).rename('Years_Since_Loss').unmask(current_year - 2000 + 1)

    fire_col = ee.ImageCollection("MODIS/061/MCD64A1").filterBounds(region).filterDate('2001-01-01', '2023-12-31').select('BurnDate')
    def get_fire_year(img):
        return ee.Image.constant(img.date().get('year')).updateMask(img.gt(0)).rename('Fire_Year').toInt()
    last_fire_year = fire_col.map(get_fire_year).max()
    years_since_fire = ee.Image.constant(current_year).subtract(last_fire_year).rename('Years_Since_Fire').unmask(current_year - 2001 + 1)

    structure_stack = ch.addBands([ch_texture, s1_vh, palsar_hv, years_since_loss, years_since_fire])

    # 4. ENVIRONMENTAL CONTEXT (Terrain, Climate, Soil, Water)
    dem = ee.Image("NASA/NASADEM_HGT/001").clip(region)
    elevation = dem.select('elevation').rename('Elevation')
    slope = ee.Terrain.slope(elevation).rename('Slope')
    aspect = ee.Terrain.aspect(elevation).rename('Aspect').multiply(math.pi).divide(180)

    worldclim = ee.Image("WORLDCLIM/V1/BIO").clip(region).select(['bio01', 'bio12']).rename(['Mean_Temp_Bio01', 'Annual_Precip_Bio12'])

    chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/PENTAD").filterBounds(region).select('precipitation')
    baseline_rain = chirps.filterDate('2000-01-01', '2020-12-31').mean()
    recent_rain = chirps.filterDate(f'{start_year}-01-01', f'{end_year}-12-31').mean()
    rain_anomaly = recent_rain.subtract(baseline_rain).divide(baseline_rain.add(0.001)).rename('Rainfall_Anomaly')

    soil = ee.Image("OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02").select('b0').rename('Soil_Clay_Surface')
    water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('seasonality').unmask(0).rename('Water_Seasonality')
    viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG").filterBounds(region).filterDate(f'{start_year}-01-01', f'{end_year}-12-31').mean().select('avg_rad').rename('Night_Lights')

    context_stack = elevation.addBands([slope, aspect.cos().rename('Northness'), aspect.sin().rename('Eastness'),
                                        worldclim, rain_anomaly, soil, water, viirs])

    # --- FINAL MERGE ---
    master_stack = phenology_stack.addBands([stability_stack, structure_stack, context_stack]).clip(region).toFloat()
    print(f"Master Stack Complete: {len(master_stack.bandNames().getInfo())} bands generated.")
    return master_stack

# Execute
# Ensure 'roi' is defined in your setup cell
master_stack = get_comprehensive_feature_stack(roi, 2020, 2023)

In [ ]:
print("Sampling 5 RANDOM pixels from the ROI to inspect feature vectors...")

debug_samples = master_stack.sample(
    region=roi,
    scale=10,
    numPixels=5,
    geometries=True
).getInfo()


features = [f['properties'] for f in debug_samples['features']]
df_debug = pd.DataFrame(features)


if df_debug.empty:
    print("WARNING: No data found. Check if your ROI has valid satellite coverage.")
else:


    print("\n--- FEATURE VECTORS (5 Random Pixels) ---")
    display(df_debug.T.style.set_properties(**{'text-align': 'right'}))

    print("\nInterpretation Check:")
    print("- NDVI_Mean should be between 0.0 and 1.0")
    print("- Elevation should be in meters (e.g., 200-300 for Delhi)")
    print("- Tree_Prob_Mean is 0.0-1.0 (Higher = More likely Forest)")

In [ ]:
print("Sampling 2,000 pixels to inspect distributions for ALL bands...")

try:

    band_names = master_stack.bandNames().getInfo()
    num_bands = len(band_names)
    print(f"Plotting histograms for {num_bands} features...")


    sample_data = master_stack.sample(
        region=roi,
        scale=SCALE,
        numPixels=2000,
        geometries=False
    ).getInfo()


    df_dist = pd.DataFrame([feat['properties'] for feat in sample_data['features']])


    cols = 5
    rows = math.ceil(num_bands / cols)


    plt.figure(figsize=(20, 4 * rows))

    for i, col in enumerate(band_names):
        if col in df_dist.columns:
            plt.subplot(rows, cols, i + 1)

            data = df_dist[col].dropna()

            if len(data) > 0:
                sns.histplot(data, kde=True, bins=30, color='skyblue', edgecolor='black')
                plt.title(col, fontsize=10)
                plt.xlabel('')
                plt.grid(True, alpha=0.3)
            else:
                plt.text(0.5, 0.5, "No Data", ha='center', va='center')
                plt.title(col)

    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Error checking distributions: {e}")

In [ ]:
# @title Phase 4: Robust Dynamic Statistical Normalization
def dynamic_preprocess(image, region, scale=10):
    print("Analyzing band statistics dynamically...")

    # 1. GET ALL BANDS
    all_bands = image.bandNames()

    # 2. CALCULATE SKEWNESS & VARIANCE
    # Uses 'reducer2' keyword to correctly combine skew and standard deviation
    sample_stats = image.reduceRegion(
        reducer=ee.Reducer.skew().combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        ),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9
    )

    stats_info = sample_stats.getInfo()

    # 3. IDENTIFY BANDS TO LOG-TRANSFORM
    # Skip indices, SAR, and metadata to avoid math errors (logs of negative numbers or decibels)
    skip_log = ['NDVI', 'NIRv', 'S1_VH', 'PALSAR_HV', 'Class', 'Phase', 't', 'constant']

    log_bands = []
    useful_bands = []

    for band in all_bands.getInfo():
        skew = stats_info.get(f'{band}_skew', 0)
        std = stats_info.get(f'{band}_stdDev', 0)

        # Drop bands with zero variation or missing data to prevent division by zero errors
        if std == 0 or std is None:
            print(f"  - Dropping {band}: No variation or invalid data.")
            continue

        is_protected = any(p in band for p in skip_log)

        # Dynamically flag for Log Transform if Skewness > 1.0 (Right-tail skew)
        if skew > 1.0 and not is_protected:
            log_bands.append(band)
            print(f"  - Log-transforming {band}: Right-skew detected ({skew:.2f})")
        else:
            if is_protected and skew > 1.0:
                print(f"  - Keeping {band} raw: Protected feature (Index/SAR/Time).")
            else:
                print(f"  - Keeping {band} raw: Normal distribution (Skew: {skew:.2f})")

        useful_bands.append(band)

    # 4. APPLY TRANSFORMS
    processed_image = image.select(useful_bands)
    for band in log_bands:
        # Create the Log version: log10(x + 1)
        log_img = processed_image.select(band).add(1).log10().rename(f'{band}_Log')

        # Add the new log band to the image
        processed_image = processed_image.addBands(log_img)

        # FIX: Remove the original raw band by selecting only the remaining band names
        remaining_bands = processed_image.bandNames().removeAll([band])
        processed_image = processed_image.select(remaining_bands)

    # 5. Z-SCORE NORMALIZATION
    print("\nScaling all features to Z-Scores...")
    final_stats = processed_image.reduceRegion(
        reducer=ee.Reducer.mean().combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        ),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9
    )

    def normalize_band(band_name):
        band_name = ee.String(band_name)
        # Use .get(key, default) to ensure if a stat is missing, the code defaults safely
        mean = ee.Number(final_stats.get(band_name.cat('_mean'), 0))
        std = ee.Number(final_stats.get(band_name.cat('_stdDev'), 1)).max(0.0001)

        return processed_image.select(band_name).subtract(mean).divide(std).rename(band_name)

    # Convert the processed ImageCollection back into a multi-band Image
    normalized_stack = ee.ImageCollection(processed_image.bandNames().map(normalize_band)).toBands()

    # Final rename to match the cleaned band names
    return normalized_stack.rename(processed_image.bandNames())

# Execute the dynamic pipeline
master_stack_norm = dynamic_preprocess(master_stack, roi, SCALE)
print(f"\nSuccess! Prepared {len(master_stack_norm.bandNames().getInfo())} bands for clustering.")
print(f"Final Clustering Bands: {master_stack_norm.bandNames().getInfo()}")

In [ ]:
# @title Phase 5: High-Res Elbow Method Sweep
from sklearn.cluster import MiniBatchKMeans

TOTAL_PIXELS = 50000
BATCH_SIZE = 5000
NUM_BATCHES = TOTAL_PIXELS // BATCH_SIZE

print(f"Fetching {TOTAL_PIXELS} pixels in {NUM_BATCHES} batches for Elbow analysis...")

all_features = []
for i in range(NUM_BATCHES):
    batch_sample = master_stack_norm.sample(region=roi, scale=SCALE, numPixels=BATCH_SIZE, seed=i).getInfo()
    all_features.extend([f['properties'] for f in batch_sample['features']])

data_for_elbow = pd.DataFrame(all_features).select_dtypes(include=[np.number])

inertia = []
K_RANGE = range(2, 21) # Checking from 2 to 20 clusters

print("Running K-Means Sweep...")
for k in K_RANGE:
    model = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3, batch_size=1024)
    model.fit(data_for_elbow)
    inertia.append(model.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_RANGE, inertia, 'bo-', linewidth=2)
plt.title('Elbow Method: Finding Optimal Ecological Zones')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Distortion)')
plt.grid(True, linestyle='--')
plt.xticks(K_RANGE)
plt.show()

print("Interpretation: Look for the 'bend' in the arm. If it's smooth, stick with k=6 for ecological diversity.")

In [ ]:
# @title Phase 6: Run SNIC Segmentation & K-Means
OPTIMAL_K = 6 # Based on typical forest classes; adjust if Elbow says otherwise

print(f"Running OBIA Clustering (k={OPTIMAL_K})...")

# 1. SNIC Segmentation (Grouping pixels into ecological stands)
snic = ee.Algorithms.Image.Segmentation.SNIC(
    image=master_stack_norm,
    size=15,          # 150m spacing for segments
    compactness=0.5,
    connectivity=8,
    neighborhoodSize=30
)

# 2. Cluster the OBJECT MEANS (The '_mean' bands)
prediction_bands = snic.select('.*_mean')
training_data = prediction_bands.sample(region=roi, scale=SCALE, numPixels=5000)
clusterer = ee.Clusterer.wekaKMeans(OPTIMAL_K).train(training_data)

object_clusters = prediction_bands.cluster(clusterer).rename('Eco_Cluster')

# Display result
Map = geemap.Map()
Map.centerObject(roi, 14)
Map.addLayer(object_clusters.randomVisualizer(), {}, "Ecological Clusters (SNIC)")
Map